# PyCRMKit 1.2.0 - 03. Domain Events, Webhooks & Communications
Ce notebook explore les mécanismes événementiels et de communication asynchrone :
- **Domain Events** : bus d'événements in-memory, souscriptions et propagation de corrélation (`correlation_id`)
- **Webhooks** : enregistrement, livraison d'événements, traçabilité des tentatives (`attempts`), rotation de secret
- **Communications (Email)** : protocoles d'envoi (`EmailProvider`), destinataires typés, contenu riche et historique des échanges

## 1. Initialisation avec Bus d'Événements et Fournisseur d'Email Simulé

In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
src_path = str(repo_root / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from pycrmkit import CRM, __version__
from pycrmkit.contacts import ContactEmail
from pycrmkit.communication.email.provider import EmailProviderResult, EmailDeliveryStatus
from pycrmkit.communication import (
    CommunicationAddress,
    CommunicationRecipient,
    CommunicationContent,
    CommunicationChannel,
)
from pycrmkit.core.references import EntityReference

# Implémentation en mémoire du protocole EmailProvider pour tester sans SMTP externe
class MockEmailProvider:
    def send(self, message):
        return EmailProviderResult(
            provider="mock-email-service",
            status=EmailDeliveryStatus.ACCEPTED,
            provider_message_id=f"msg-mock-{str(message.intent_id)[:8]}",
        )

crm = CRM.memory(email_provider=MockEmailProvider()).with_context(
    actor_id="event-integrator",
    correlation_id="async-session-003"
)

print(f"✅ PyCRMKit version : {__version__}")

✅ PyCRMKit version : 1.2.0


## 2. Souscription aux Événements de Domaine
Nous écoutons l'événement `contact.created` pour auditer automatiquement les créations.

In [2]:
captured_events = []

def on_contact_created(event):
    captured_events.append(event)
    print(f"⚡ Événement intercepté : [{event.type}] sur l'agrégat {event.aggregate_id}")
    print(f"   Corrélation : {event.correlation_id} | Acteur : {event.actor_id}")

crm.events.subscribe("contact.created", on_contact_created)

# Déclenchement de l'événement lors de la création d'un contact
contact = crm.contacts.create(
    first_name="Grace",
    last_name="Hopper",
    emails=(ContactEmail("grace.hopper@navy.mil", is_primary=True),),
)

assert len(captured_events) == 1
print(f"✅ Événement stocké : {captured_events[0].type}")

⚡ Événement intercepté : [contact.created] sur l'agrégat ce835506-70bb-4b97-98a4-4e46d69b7871
   Corrélation : async-session-003 | Acteur : event-integrator
✅ Événement stocké : contact.created


## 3. Webhooks : Enregistrement, Livraison et Rotation de Clé
Enregistrement d'un endpoint webhook pour notifier des systèmes tiers en temps réel.

In [3]:
# 1. Enregistrement d'un abonnement webhook
sub = crm.webhooks.register(
    url="https://api.external-erp.io/v1/crm-webhooks",
    events=("contact.created", "opportunity.created"),
    signing_secret="whsec_initial_secret_123456",
)
print(f"🔔 Webhook enregistré : ID {sub.id}")
print(f"   URL : {sub.url}")
print(f"   Secret : {sub.signing_secret[:12]}...")

# 2. Simulation de livraison de l'événement capturé
deliveries = crm.webhooks.deliver(captured_events[0])
print(f"📤 Tentative de livraison effectuée : {len(deliveries)} livraison(s)")
for d in deliveries:
    print(f"   Abonnement {d.subscription_id} -> État : {d.state}")

# 3. Rotation de clé secrète
updated_sub = crm.webhooks.rotate_secret(
    sub.id,
    signing_secret="whsec_rotated_secret_987654"
)
print(f"🔑 Secret rotaté avec succès : {updated_sub.signing_secret[:12]}...")

🔔 Webhook enregistré : ID 948d6948-4094-45dc-a8ea-3670569097d7
   URL : https://api.external-erp.io/v1/crm-webhooks
   Secret : whsec_initia...
📤 Tentative de livraison effectuée : 1 livraison(s)
   Abonnement 948d6948-4094-45dc-a8ea-3670569097d7 -> État : retry_scheduled
🔑 Secret rotaté avec succès : whsec_rotate...


## 4. Envoi d'Email et Historique de Communication
Création et expédition d'un message transactionnel rattaché au contact.

In [4]:
contact_ref = EntityReference("contact", contact.id)
recipient_addr = CommunicationAddress(channel=CommunicationChannel.EMAIL, value="grace.hopper@navy.mil")
sender_addr = CommunicationAddress(channel=CommunicationChannel.EMAIL, value="contact@pycrmkit.io")

recipients = (CommunicationRecipient(address=recipient_addr, display_name="Grace Hopper"),)
content = CommunicationContent(
    subject="Bienvenue dans l'écosystème PyCRMKit",
    text_body="Bonjour Grace, votre compte a été initialisé avec succès.",
    html_body="<p>Bonjour <strong>Grace</strong>,<br>Votre compte a été initialisé avec succès.</p>",
)

email_record = crm.email.send(
    to=recipients,
    sender=sender_addr,
    content=content,
    references=(contact_ref,),
    idempotency_key="welcome-email-hopper-001",
)

print(f"✉️ Email envoyé avec succès : ID {email_record.id}")
print(f"   Objet : {email_record.subject}")
print(f"   Statut de remise : {email_record.delivery_status}")

# Consultation de l'historique d'emails du contact
history = crm.email.for_contact(contact.id)
print(f"📜 Historique de communication pour {contact.display_name} : {history.total} message(s)")

✉️ Email envoyé avec succès : ID 6216f86d-edc3-4e18-a1dd-dc04bfd65d19
   Objet : Bienvenue dans l'écosystème PyCRMKit
   Statut de remise : sent
📜 Historique de communication pour Grace Hopper : 1 message(s)


## 5. Assertions de Validation du Notebook

In [5]:
assert len(captured_events) >= 1
assert str(captured_events[0].type) == "contact.created"
assert updated_sub.signing_secret == "whsec_rotated_secret_987654"
assert len(deliveries) >= 1
assert history.total == 1
assert history.items[0].subject == "Bienvenue dans l'écosystème PyCRMKit"

print("✨ TOUS LES TESTS DU NOTEBOOK 03 (EVENTS, WEBHOOKS & EMAIL) SONT VALIDÉS AVEC SUCCÈS !")

✨ TOUS LES TESTS DU NOTEBOOK 03 (EVENTS, WEBHOOKS & EMAIL) SONT VALIDÉS AVEC SUCCÈS !
